# MoTA-SANKALP: fine-tune YOLOv8-nano on Kaggle datasets + own certificates

Run this on **Kaggle Notebooks**. Settings: Accelerator = **GPU T4**, Internet = **On**.

Add these inputs (right panel, *Add Input*):

| Input | Used for |
|-------|----------|
| `victordibia/signverod` | signatures -> `tehsildar_sign` |
| `aravindnagarajan/stamp-data` | stamps -> `official_seal` |
| `rtatman/stamp-verification-staver-dataset` (optional, very large) | stamps -> `official_seal` |
| your private dataset **mota-ml-code** (upload the `ml/` folder) | these scripts |
| your private dataset **mota-real-certs** (Label Studio YOLO export) | all 5 classes + the TEST split |

`income_box` and `st_box` exist in NO public dataset. They come only from `mota-real-certs` (and optional synthetic data).
Keep `mota-real-certs` **private** and upload only consented, masked images.

Run the cells in order. Cell 4 prints the real class names. Check them before running cell 5.

In [ ]:
# 1. GPU check and list attached inputs
!nvidia-smi --query-gpu=name,memory.total --format=csv
!find /kaggle/input -maxdepth 4 -type d | head -60

In [ ]:
# 2. Locate inputs (Kaggle mount paths can differ, so search by name)
import os, shutil, subprocess

def find_input(name):
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.count(os.sep) > 6:
            dirs[:] = []
            continue
        if os.path.basename(root) == name:
            return root
    return None

def find_file(filename):
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return root
    return None

ML_SRC    = find_file('train_yolo.py')
SIGNVEROD = find_input('signverod')
STAMPS    = find_input('stamp-data')
STAVER    = find_input('stamp-verification-staver-dataset')
REAL      = find_input('mota-real-certs')
print('ml code  :', ML_SRC)
print('signverod:', SIGNVEROD)
print('stamps   :', STAMPS)
print('staver   :', STAVER)
print('own certs:', REAL)

ML = '/kaggle/working/ml'
shutil.copytree(ML_SRC, ML, dirs_exist_ok=True)
os.chdir(ML)

In [ ]:
# 3. Install the pinned ultralytics version
!pip install -q ultralytics==8.4.157

In [ ]:
# 4. INSPECT every dataset. Read the class names and formats printed here.
for path in [SIGNVEROD, STAMPS, STAVER, REAL]:
    if path:
        print('=' * 70)
        !python inspect_dataset.py "{path}"

### Before cell 5: check the mapping

The `--map` values below assume the class names are `signature` (SignverOD) and `stamp` (Stamp-Data).
If cell 4 printed different names, edit the `--map` arguments to match **exactly**. Names are case-insensitive.
If StaVer shows a mask folder rather than boxes, use the masks command and fill in the two folders cell 4 printed.

In [ ]:
# 5. Convert Kaggle datasets to the canonical 5-class layout
C = '/kaggle/working/conv'
!python convert_to_yolo.py --src "{SIGNVEROD}" --out {C}/signverod --prefix sv --map signature=tehsildar_sign --limit 1500
!python convert_to_yolo.py --src "{STAMPS}" --out {C}/stamps --prefix st --map stamp=official_seal

# StaVer (only if attached and cell 4 showed masks). Fill in the two folders:
# !python convert_to_yolo.py --format masks --images "<scan folder>" --masks "<mask folder>" \
#     --out {C}/staver --prefix sv2 --mask-class official_seal --limit 400

In [ ]:
# 6. Copy own certificates to a writable folder and auto-label QR codes
REAL_W = '/kaggle/working/real'
shutil.copytree(REAL, REAL_W, dirs_exist_ok=True)
!python autolabel_qr.py --dir {REAL_W}

In [ ]:
# 7. Build the dataset. The test split comes ONLY from your own certificates.
SOURCES = f'{C}/signverod {C}/stamps'
if os.path.isdir(f'{C}/staver'):
    SOURCES += f' {C}/staver'
!python prepare_dataset.py --src {SOURCES} --test-from {REAL_W} --out /kaggle/working/datasets/mota_mix

### Before cell 8: read the class table printed above

Every class needs a reasonable number of **train** boxes and some **test** boxes.
If `income_box` or `st_box` is near zero, annotate more of your own certificates. Training will not fix missing data.

In [ ]:
# 8. Fine-tune.
# Option A: from COCO weights (default)
!python train_yolo.py --data /kaggle/working/datasets/mota_mix/data.yaml --name real_v1 --epochs 150 --batch 16 --onnx

# Option B: continue from your synthetic model (upload best.pt as a dataset first)
# !python train_yolo.py --data /kaggle/working/datasets/mota_mix/data.yaml \
#     --weights /kaggle/input/<your-synthetic-weights>/best.pt --lr0 0.0005 --freeze 10 \
#     --name real_finetune_v1 --epochs 100 --onnx

In [ ]:
# 9. Package results for download (Output tab after the session ends)
!cat runs_mota/real_v1/test_metrics.json
!cd runs_mota && zip -qr /kaggle/working/mota_model_real_v1.zip real_v1/weights real_v1/test_metrics.json real_v1/*.png real_v1_test
print('Download /kaggle/working/mota_model_real_v1.zip from the Output panel')